# Preprocessing

In [ ]:
import os
import numpy as np
import pandas as pd
import main_dataset_utils as mdu

In [ ]:
MAIN_DATASET_PATH = "" # ukb main dataset path (.csv format)
SAVE_DIR = "./data" # directory to save processed files
withdrawals_path = "" # ukb withdrawals file path

specs = [
    # ----- sample filtering -----
    # self-reported sex; Data-coding 9 (0=female, 1=male)
    ('srsex', 31, 'numeric'), 
    # genetic sex; Data-coding 9 (0=female, 1=male)
    ('genesex', 22001, 'numeric'),
    # ethnicity; Data-coding 1002 (1=caucasian) 
    ('ethnic', 22006, 'binary'), 
    # sex chromosome aneuploidy; Data-coding 1 (1=yes)
    ('sca', 22019, 'numeric'), # numeric to keep NaNs as non-sca are NaN
    # relatedness; Data-coding 1 (1=yes)
    ('in_genepcs', 22020, 'binary'), 

    # ----- phenotype adjustment / covars ----- 
    ('age', 21022, 'numeric'), # age at recruitment in yrs
    
    # ----- phenotype(s) ----- 
    ('height', 50, 'numeric'), # standing height in cm
]

Load and parse main dataset

In [ ]:
# load main dataset with all required fields
field_ids = set([field_id for _, field_id, _ in specs])
print(f"Target fields: {field_ids}")
print(f"Loading main dataset...")
main_raw_df = mdu.load_main_dataset(MAIN_DATASET_PATH, field_ids)
print(f"Loaded main dataset with {main_raw_df.shape[0]} samples and "
      f"{main_raw_df.shape[1]} columns across {len(field_ids)} fields.")
# parse dataset (clean data and format col names)
main_df = mdu.parse_main_dataset(main_raw_df, specs)
# print(f"After parsing, {main_df.shape[1]} columns remain.")
print(f"\nParsing summary:")
for colidx, col in enumerate(main_df.columns):
    print(f"{str(colidx+1)+'. '+col:<20} (dtype={main_df[col].dtype}, n_unique={main_df[col].nunique(dropna=False)}, "
          f"n_nan={main_df[col].isna().sum()}, n_nan_pct={main_df[col].isna().mean()*100:.2f}%)")

Filter samples (main dataset sample QC)

In [ ]:
# filter main dataset based on defined criteria
eid = main_df['eid']
drops = set(pd.read_csv(
    withdrawals_path, header=None, 
    dtype={0: int}
)[0].tolist())
main_qc_mask = (
    ~eid.isin(drops) &                          # is not withdrawn
    (main_df['srsex'] == main_df['genesex']) &  # is sex concordant
    (main_df['ethnic'] == 1) &                  # is caucasian
    (main_df['sca'].isna()) &                   # is not sex-chrom aneuploidy
    (main_df['in_genepcs'] == 1)                # is not related
)
main_qc_mask = main_qc_mask.replace({pd.NA: False})
assert len(main_qc_mask) == main_raw_df.shape[0]
print(f"\nMain dataset sample QC summary:\n{main_qc_mask.value_counts(dropna=False)}")

Phenotypes

In [ ]:
pheno_names = ['height'] # phenotype names (as found in specs)
pheno_df = main_df[['eid'] + pheno_names].copy()
print(f"phenotype(s) info:")
for col in pheno_df.columns:
    if col != 'eid':
        print(f"{col:<16} dtype={pheno_df[col].dtype}, "
              f"n_unique={pheno_df[col].nunique(dropna=False)}, "
              f"n_missing={pheno_df[col].isna().sum()}, "
              f"missing_pct={pheno_df[col].isna().sum()/pheno_df.shape[0]:.2%}")
print(f"\nphenotype(s) stats:\n{pheno_df.describe()}")

nonmissing_pheno_mask = pheno_df[pheno_names].notna()# .all(axis=1)
print(f"\nPhenotype non-missing mask:\n"
      f"{nonmissing_pheno_mask.value_counts(dropna=False)}")

Mask: keep samples passing main dataset QC and non-missing phenotype labels

In [ ]:
# if there's only one phenotype, .squeeze() works here;
# if there's more than one phenotype, use either:
#   .all(axis=1) : keep sample if all phenotype labels are present.
#   .any(axis=1) : keep sample if at least one phenotype label is present.
keep_mask = main_qc_mask & nonmissing_pheno_mask.squeeze()
eid_keep = eid[keep_mask]

# save dataframe and use it w/ PLINK --keep <keep_file> option
plink_keep_df = pd.DataFrame({'FID': eid_keep, 
                              'IID': eid_keep})
print(f"QC + nonmissing phenotype mask:\n"
      f"{keep_mask.value_counts(dropna=False)}")

Save filtered samples, phenotypes, covariates

In [ ]:
plink_pheno_df = pd.DataFrame({
    'FID': eid,
    'IID': eid,
    'height': main_df['height']
})
plink_covars_df = pd.DataFrame({
    'FID': eid,
    'IID': eid,
    'age': main_df['age'],
    'sex': main_df['genesex']
})
# save PLINK files
plink_keep_path = os.path.join(SAVE_DIR, 'main_keep_ids.txt')
plink_pheno_path = os.path.join(SAVE_DIR, 'height.pheno')
plink_covars_path = os.path.join(SAVE_DIR, 'main_covars.covar')

os.makedirs(SAVE_DIR, exist_ok=True)
plink_keep_df.to_csv(plink_keep_path, sep='\t', index=False, header=True)
print(f"Saved keep IDs to:\n{plink_keep_path}")
plink_pheno_df.to_csv(plink_pheno_path, sep='\t', index=False, header=True)
print(f"Saved phenotype data to:\n{plink_pheno_path}")
plink_covars_df.to_csv(plink_covars_path, sep='\t', index=False, header=True)
print(f"Saved covariates to:\n{plink_covars_path}")


## PLINK: Genotype Merge & QC

Merge individual autosomal genotype calls into a single merged dataset (PLINK v1.9)

```bash
plink --merge-list ./merge_list_all_autosomes.txt --make-bed  --out ukb_c1-22
```

where `merge_list_all_autosomes.txt` contains a list of the individual chromosome-level PLINK binaries (bim/fam/bed) to merge into a single bim/fam/bed fileset. 


**PLINK QC Step:**

Start with samples that passed main dataset filtering from the step above (sample qc + pheno missingness filtering) using `--keep main_keep_ids.txt`.

```bash 
# PLINK genotype QC
plink2 --bfile ukb_c1-22 \
  --autosome \
  --keep main_keep_ids.txt \
  --geno 0.1 \
  --hwe 1e-15 \
  --mac 100 \
  --maf 0.01 \
  --mind 0.1 \
  --write-samples --write-snplist \
  --make-bed \
  --out ukb_c1-22_qc
```

## Train-Val-Test Splits

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
samples_path = './data/ukb_c1-22_qc.id' # from PLINK QC step above
covars_path = "./data/main_covars.covar"
train_frac, val_frac, test_frac = 0.8, 0.1, 0.1
rs = 1234  # random state
SAVE_DIR = './data'
SAVE_DIR = os.path.abspath(SAVE_DIR)

In [ ]:
# Train/Val/Test split
def train_val_test_split(eid_series, train_frac, val_frac, test_frac, 
                          random_state=None, stratify=True):
    assert np.sum([train_frac, val_frac, test_frac]) == 1, \
        f"Splits fractions must sum to 1. " + \
        f"Found {train_frac:.2f}+{val_frac:.2f}+{test_frac:.2f} = " + \
        f"{train_frac+val_frac+test_frac:.1f}"
    
    trainval, test = train_test_split(
        eid_series.index.values, stratify=eid_series if stratify else None, 
        test_size=test_frac, random_state=random_state
    )
    adjusted_val_frac = val_frac / (1 - test_frac)
    train, val = train_test_split(
        trainval, stratify=eid_series.loc[trainval] if stratify else None,
        test_size=adjusted_val_frac, random_state=random_state
    )
    return train, val, test

eid = pd.read_csv(samples_path, sep='\t', dtype=str)
eid.rename(columns={'#FID': 'FID'}, inplace=True)

covars_df = pd.read_csv(covars_path, sep='\t', dtype={'FID': str, 'IID': str})
covars_df = covars_df.loc[covars_df['IID'].isin(eid['IID']), :]

# keep the sex distribution consistent across splits (relative to the full dataset)
partition_series = pd.Series(covars_df['sex'].values, index=covars_df['IID'])
eid_train, eid_val, eid_test = train_val_test_split(
    partition_series, train_frac, val_frac, test_frac, 
    random_state=rs, stratify=True
)

# sort participant IDs within each split
eid_train = sorted(eid_train)
eid_val = sorted(eid_val)
eid_test = sorted(eid_test)

# Create plink-style dataframes for each split; columns [FID, IID]
plink_eid_train = pd.DataFrame({'FID':eid_train, 'IID':eid_train})
plink_eid_val = pd.DataFrame({'FID':eid_val, 'IID':eid_val})
plink_eid_test = pd.DataFrame({'FID':eid_test, 'IID':eid_test})

assert len(plink_eid_train)+len(plink_eid_val)+len(plink_eid_test)==len(eid), \
    "n_train + n_val + n_test != n_total"

print(f"Train/val/test splits created (random_state={rs}):")
print(f"Total number of samples considered for partitioning: {len(partition_series):,}")
print(f"Partition sizes:\n"
      f"  n_train: {len(plink_eid_train)}\n"
      f"  n_val:   {len(plink_eid_val)}\n"
      f"  n_test:  {len(plink_eid_test)}\n")

print(f"Train-val contamination: "
      f"{len(set(plink_eid_train['IID']).intersection(plink_eid_val['IID'])) > 0}")
print(f"Train-test contamination: "
      f"{len(set(plink_eid_train['IID']).intersection(plink_eid_test['IID'])) > 0}")
print(f"Val-test contamination: "
      f"{len(set(plink_eid_val['IID']).intersection(plink_eid_test['IID'])) > 0}")

eid_train_path = os.path.join(SAVE_DIR, f'train_rs{rs}.id')
eid_val_path = os.path.join(SAVE_DIR, f'val_rs{rs}.id')
eid_test_path = os.path.join(SAVE_DIR, f'test_rs{rs}.id')
os.makedirs(SAVE_DIR, exist_ok=True)
plink_eid_train.to_csv(eid_train_path, sep='\t', index=False, header=True)
plink_eid_val.to_csv(eid_val_path, sep='\t', index=False, header=True)
plink_eid_test.to_csv(eid_test_path, sep='\t', index=False, header=True)
print(f"\nTrain sample IDs saved to:      {os.path.abspath(eid_train_path)}")
print(f"Validation sample IDs saved to: {os.path.abspath(eid_val_path)}")
print(f"Test sample IDs saved to:       {os.path.abspath(eid_test_path)}")

## Phenotype Adjustment

- Fit ordinary least squares (OLS) model for phenotype using age and sex as predictors (training set only).
- Compute residuals (i.e., portion of height variation unexplained by age and sex).
- Standardize residuals with z-score (and/or inverse-normal transform)

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import rankdata, norm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pheno_path = "./data/height.pheno"
pheno_name = "height"
covars_path = "./data/main_covars.covar"
train_ids_path = "./data/train_rs1234.id"
SAVE_DIR = './data'
SAVE_DIR = os.path.abspath(SAVE_DIR)

In [ ]:
# load necessary data
covars = pd.read_csv(covars_path, sep='\t', 
                     dtype={'IID':str})

phenos = pd.read_csv(pheno_path, sep='\t', 
                     dtype={'IID':str})
eid_train = pd.read_csv(train_ids_path, sep='\t', 
                        usecols=['IID'], dtype={'IID':str})

# ensure appropriate data types
covars['sex'] = covars['sex'].astype('category')
covars['age'] = covars['age'].astype(float)
phenos[pheno_name] = phenos[pheno_name].astype(float)

# get a training set dataframe with training samples only
train_df = pd.merge(eid_train, covars, on='IID', how='left')
train_df = pd.merge(train_df, phenos, on='IID', how='left')

# extract the X and y training data
X_train = train_df.loc[:, ['sex', 'age']]

# add a constant to the X vars (intercept)
X_train = sm.add_constant(X_train)
y_train = train_df.loc[:, pheno_name]

# fit the linear regression model on training set
model = sm.OLS(y_train, X_train).fit()
betas_df = model.params.to_frame().reset_index()

# mean and standard deviation of residuals
training_mean = model.resid.mean()
training_std = model.resid.std()
residuals_stats_df=pd.DataFrame([('mean', training_mean), 
                                 ('std', training_std)])

print(f"Linear regression model coefficients:\n{betas_df}\n")
print(f"Residuals stats (mean, std):\n{residuals_stats_df}\n")

adjust phenotype for all samples and standardize

In [ ]:
def invnorm_transform(x):
    # x is a Series of residuals (NaNs are ok)
    non_na = x.dropna()
    if non_na.empty:
        return pd.Series(np.nan, index=x.index)
    ranks = rankdata(non_na, method='average')
    u = (ranks - 0.5) / len(non_na) # map to (0,1)
    y = pd.Series(norm.ppf(u), index=non_na.index)
    out = pd.Series(np.nan, index=x.index)
    out.loc[non_na.index] = y
    return out

# combine covars and phenos for the entire dataset
all_data_df = pd.merge(covars, phenos, on='IID', how='inner')

# prepare the X and y data for prediction (all samples) 
X_all = all_data_df.loc[:, ['sex', 'age']]
X_all = sm.add_constant(X_all)

# predict phenotype on all samples using fitted OLS model
all_data_df[f'{pheno_name}_pred'] = model.predict(X_all)

# calculate residuals (i.e., adjusted phenotypes) for all samples
all_data_df[f'{pheno_name}_adj'] = all_data_df[pheno_name] - all_data_df[f'{pheno_name}_pred']

# z-score standardize the residuals for the entire dataset
all_data_df[f'{pheno_name}_adj_z'] = (all_data_df[f'{pheno_name}_adj'] - training_mean) / training_std

# inverse-normal transform the residuals for the entire dataset 
# (as an alternative option to z-score)
all_data_df[f'{pheno_name}_adj_int'] = invnorm_transform(all_data_df[f'{pheno_name}_adj'])

# format the data and save to files
cols_to_merge = [
    pheno_name,
    f'{pheno_name}_pred',
    f'{pheno_name}_adj',
    f'{pheno_name}_adj_z',
    f'{pheno_name}_adj_int',
]
plink_adjusted_phenos = (
    pd.DataFrame({'FID': all_data_df['IID'].astype(str),
                  'IID': all_data_df['IID'].astype(str)})
    .merge(all_data_df.loc[:, cols_to_merge], how='inner',
           left_index=True, right_index=True)
)

print(f"Adjusted phenotypes summary:\n{plink_adjusted_phenos.describe()}\n")

phenos_path = os.path.join(SAVE_DIR, 'height_adj.pheno')
plink_adjusted_phenos.to_csv(phenos_path,
                             sep='\t', index=False,
                             na_rep='NA', header=True)
betas_path = os.path.join(SAVE_DIR, 'height_adj_betas.txt')
betas_df.to_csv(betas_path,
                 sep='\t', index=False,
                 na_rep='NA', header=True)
residuals_stats_path = os.path.join(SAVE_DIR, 'height_adj_residuals_stats.txt')
residuals_stats_df.to_csv(residuals_stats_path,
                           sep='\t', index=False,
                           na_rep='NA', header=True)

print(f"Adjusted phenotypes saved to: {os.path.abspath(phenos_path)}")
print(f"Model coefficients saved to: {os.path.abspath(betas_path)}")
print(f"Residuals stats saved to: {os.path.abspath(residuals_stats_path)}")

plot histograms (raw phenotypes, adjusted phenotypes, & normalized adjusted phenotypes)

In [ ]:
pheno_hist_df = all_data_df.loc[:, ['sex', 
                                    pheno_name, f'{pheno_name}_adj', 
                                    f'{pheno_name}_adj_z', 
                                    f'{pheno_name}_adj_int']].copy()
target_cols = ['height', 'height_adj', 'height_adj_z', 'height_adj_int']
subplot_titles = ['Original height', 'Adjusted height', 
                  'Adjusted height (z-score)', 'Adjusted height (inverse-normal)']
# remap sex codes for plotting (0=female, 1=male)
pheno_hist_df['sex_label'] = pheno_hist_df['sex'].map({0: 'Female', 1: 'Male'})
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for ax, col, title in zip(axes, target_cols, subplot_titles):
    sns.histplot(
        data=pheno_hist_df,
        x=col,
        hue='sex_label',
        kde=True,
        ax=ax,
        element='step',
        stat='density',
        common_norm=False,
        palette='Set2'
    )
    stats = (
        pheno_hist_df.groupby('sex_label', observed=True)[col]
        .agg(['mean', 'std'])
        .round(2)
    )
    ymax = ax.get_ylim()[1]
    text_y = ymax * 0.92  # starting vertical position
    for i, (group, row) in enumerate(stats.iterrows()):
        ax.text(
            0.02, text_y - i * (ymax * 0.06),  # offset by 10% of y-range
            f"{group}: μ={row['mean']:.2f}, σ={row['std']:.2f}",
            transform=ax.get_yaxis_transform(),
            fontsize=11,
            fontweight='bold',
            color=sns.color_palette('Set2')[i],
            ha='left',
            va='top'
        )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(title.lower())
    ax.set_ylabel("Density")
plt.tight_layout()
plt.show()

## GWAS

```bash
plink2 --bfile ./data/ukb_c1-22_qc \
       --keep ./data/train_rs1234.id \
       --glm allow-no-covars --variance-standardize \
       --pheno ./data/height_adj.pheno \
       --pheno-name height_adj_z \
       --out ./data/gwas
```

## Minor Allele Frequency (MAF) report

**train only**

```bash
plink2 --bfile ./data/ukb_c1-22_qc \
       --keep ./data/train_rs1234.id \
       --freq --out ./data/maf_train_rs1234
```

**all samples**

```bash
plink2 --bfile ./data/ukb_c1-22_qc \
       --freq --out ./data/maf_all
```